# Bridge Nonlinearity with an Initial Offset

**A quarter-bridge measured after a software zero is only as linear as its starting point.** When the bridge already sits away from resistive balance — because of preload, a bonding shift, or a large static strain — the incremental reading you take after taring inherits a nonlinearity that depends on *where the bridge started*, not merely on how much strain you add.

*Companion notebook to Chapter 27 — Bridge Configurations and Selection.*

This notebook reproduces the initial-imbalance argument the chapter makes in *Initial Imbalance and Software Zero* (Eq. 27.7). We sweep the initial offset of a single-active-gage quarter-bridge and watch how the nonlinearity of a *software-zeroed* strain increment grows as the bridge drifts from balance.

In [ ]:
# !pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt

# This notebook is fully deterministic: every curve is a closed-form function
# of the bridge equations, so there is no randomness to seed.

## The exact quarter-bridge output

Chapter 27 gives the *exact* output of a single-active-gage quarter-bridge as a dimensionless voltage ratio — not the linear approximation, but the full expression that keeps the shifting arm currents (Eq. 27.4):

$$\frac{V_\text{out}}{V_\text{ex}} = \frac{GF\,\varepsilon\times10^{-6}}{4 + 2\,GF\,\varepsilon\times10^{-6}}$$

The numerator is the fractional resistance change $\Delta R/R = GF\,\varepsilon$; the denominator's second term, $2\,GF\,\varepsilon\times10^{-6}$, is exactly what the linear approximation discards. In words: as strain grows, the arm currents no longer hold constant, and the output bends gently below the straight line the instrument assumes.

We encode this exact ratio once, in a small function, and build everything else on top of it.

In [ ]:
# --- Fixed measurement parameters -------------------------------------------
GAGE_FACTOR = 2.0                  # dimensionless; typical constantan foil gage
MICROSTRAIN = 1e-6                 # converts microstrain (mu-epsilon) to strain
MICROSTRAIN_PER_PERCENT = 10_000   # 1% strain = 10,000 microstrain

# --- Offset sweep -----------------------------------------------------------
MAX_OFFSET_MU = 50_000             # sweep the initial imbalance from 0 to 5% strain
N_OFFSET_POINTS = 500              # resolution of the offset sweep

# Strain increments applied *after* the software zero, expressed in percent.
# 0.01% = 100 microstrain up to 10% = 100,000 microstrain: elastic to deep post-yield.
INCREMENTS_PCT = [0.01, 0.1, 1.0, 10.0]


def bridge_output(strain_mu, gage_factor=GAGE_FACTOR):
    """Exact normalized quarter-bridge output V_out / V_ex (Eq. 27.4).

    Parameters
    ----------
    strain_mu : float or np.ndarray
        True surface strain at the active gage, in microstrain (mu-epsilon).
    gage_factor : float
        Gage factor GF (dimensionless).

    Returns
    -------
    float or np.ndarray
        Dimensionless output ratio V_out / V_ex, including the second-order
        term the linear approximation omits.
    """
    dr_over_r = gage_factor * strain_mu * MICROSTRAIN  # fractional resistance change dR/R
    return dr_over_r / (4 + 2 * dr_over_r)

## The error in a software-zeroed increment

In practice the engineer rarely starts from a perfectly balanced bridge. A gage bonded to a preloaded part, or a bridge that shifted during installation, sits at some **initial offset** before measurement begins. The usual remedy is a *software zero*: record the offset voltage and subtract it from every later reading.

The tare removes the number, but not the physics. The instrument still converts the *change* in output to strain with the linear rule $\Delta\varepsilon_i = (4/GF)\,\Delta(V_\text{out}/V_\text{ex})$ — and because the exact output curves away from that line, the increment reads low by an amount that depends on how far the bridge sits from balance. This is exactly the effect the chapter corrects with the closed-form Eq. 27.7.

The function `nonlinearity_error` below computes that error directly. It evaluates the *exact* output at the initial offset and again at the offset-plus-increment, takes the difference (the software-zeroed reading), converts it back to strain with the linear rule, and compares it to the true increment. We then sweep the offset and plot the error for several increment sizes.

In [ ]:
def nonlinearity_error(offset_mu, increment_mu, gage_factor=GAGE_FACTOR):
    """Percent nonlinearity error of a software-zeroed strain increment.

    The bridge is taken to sit at ``offset_mu`` of true strain when the
    software zero is applied; a true strain ``increment_mu`` is then added.
    The instrument reports the increment via the linear quarter-bridge rule,
    which understates the true increment because the exact output (Eq. 27.4)
    curves below the straight line the instrument assumes.

    Parameters
    ----------
    offset_mu : float or np.ndarray
        Initial bridge offset expressed as equivalent true strain (microstrain).
    increment_mu : float
        True strain increment applied after the software zero (microstrain).
    gage_factor : float
        Gage factor GF (dimensionless).

    Returns
    -------
    float or np.ndarray
        Percent error of the indicated increment relative to the true
        increment (negative = reads low, as expected for tension).
    """
    v_initial = bridge_output(offset_mu, gage_factor)
    v_final = bridge_output(offset_mu + increment_mu, gage_factor)
    delta_v = v_final - v_initial  # the software-zeroed change in output ratio

    # Instrument's linear back-conversion: delta_e_i = (4 / GF) * delta(Vout/Vex)
    indicated_increment_mu = (4 / gage_factor) * delta_v / MICROSTRAIN

    return (indicated_increment_mu / increment_mu - 1) * 100


# Sweep the initial offset from a perfectly balanced bridge out to 5% strain.
offsets_mu = np.linspace(0, MAX_OFFSET_MU, N_OFFSET_POINTS)

fig, ax = plt.subplots(figsize=(10, 6))

for increment_pct in INCREMENTS_PCT:
    increment_mu = increment_pct * MICROSTRAIN_PER_PERCENT
    errors_pct = nonlinearity_error(offsets_mu, increment_mu)
    # Plot the offset on the x-axis in percent strain (mu-epsilon / 10,000).
    ax.plot(offsets_mu / MICROSTRAIN_PER_PERCENT, errors_pct,
            label=f'Increment: {increment_pct}% strain')

ax.set_title('Effect of Initial Offset on Quarter-Bridge Nonlinearity')
ax.set_xlabel('Initial Offset (% strain)')
ax.set_ylabel('Nonlinearity Error (%)')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.legend()

fig.tight_layout()
plt.show()

## Where to take this next

- **Add the closed-form correction (Eq. 27.7) and confirm it flattens the curves.** Implement $\Delta\varepsilon = 4\,\Delta\varepsilon_i \big/ \big[(2 - GF\,\varepsilon_{0,i}\times10^{-6})(2 - GF\,(\varepsilon_{0,i}+\Delta\varepsilon_i)\times10^{-6})\big]$, feed each curve's indicated increment back through it, and verify the residual error collapses to zero — a direct check that the chapter's correction actually works.
- **Compare configurations.** Repeat the sweep for the axial/Poisson half-bridge (also nonlinear) and for a bending half-bridge (current-symmetric, ideally linear). Overlaying them shows *why* the chapter recommends opposing-arm bridges for precision work: their calibration equation stays exact even far from balance.
- **Turn the offset into a tolerance.** Fix an error budget — say 0.5% — and solve for the largest initial offset each increment size can tolerate before the software-zeroed reading needs the Eq. 27.7 correction. That gives a practical rule for deciding when to physically rebalance the bridge rather than simply tare it.